# CMPE 258 Team 7: Check-In 2 reproducible run (Colab T4)

**Robust Neural Image Watermarking with Failure-Guided Attack Sampling (FGAS)**

This notebook is the team's canonical environment. Teammates use different machines, so every reported number comes from this notebook on a **Colab T4 GPU** with the fixed seed in `configs/baseline.yaml`.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.

What it does:
1. Clones the repo and installs dependencies. Colab's preinstalled PyTorch is used.
2. Downloads DIV2K HR and caches the zips on Google Drive. Then it prepares the 600-train / 100-val subset.
3. Runs the unit tests.
4. Trains the uniform-sampling baseline (Experiments A and B).
5. Runs the standalone attack-wise evaluation, which gives Table 1 data.
6. Computes the FGAS probability update from real validation results (Experiment C, Table 2 data).
7. Runs a short FGAS training run. It shows the trainer consuming the new probabilities.
8. Copies everything to `results/` and to the team Google Drive folder.

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


In [2]:
# Team Drive folder: checkpoints, run logs and the DIV2K zip cache go here.
# Change DRIVE_DIR if the shared team folder lives somewhere else in your Drive.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/CMPE258_Team7'
import os
os.makedirs(f'{DRIVE_DIR}/cache', exist_ok=True)

Mounted at /content/drive


In [3]:
REPO_URL = 'https://github.com/TaehunLeee5/CMPE258_Team7.git'
BRANCH = 'main'
%cd /content
!test -d CMPE258_Team7 || git clone -b {BRANCH} {REPO_URL}
%cd /content/CMPE258_Team7
!git pull --ff-only && git log -1 --oneline
!pip install -q -r requirements.txt
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))"

/content
Cloning into 'CMPE258_Team7'...
remote: Enumerating objects: 89, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 89 (delta 31), reused 83 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (89/89), 42.99 KiB | 2.69 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/CMPE258_Team7
Already up to date.
3e9b571 (HEAD -> main, origin/main, origin/HEAD) Created using Colab
torch 2.11.0+cu130 | cuda True | Tesla T4


## 1. Data: DIV2K

In [4]:
!python scripts/download_div2k.py --config configs/baseline.yaml --cache-dir {DRIVE_DIR}/cache
!python scripts/prepare_div2k.py --config configs/baseline.yaml

   3367.0 / 3367.0 MB
Extracting /content/drive/MyDrive/CMPE258_Team7/cache/DIV2K_train_HR.zip -> data/DIV2K
train: 800 images in data/DIV2K/DIV2K_train_HR
Extracting /content/drive/MyDrive/CMPE258_Team7/cache/DIV2K_valid_HR.zip -> data/DIV2K
valid: 100 images in data/DIV2K/DIV2K_valid_HR
train: 100% 600/600 [02:59<00:00,  3.34it/s]
val: 100% 100/100 [00:28<00:00,  3.47it/s]
Prepared 600 train / 100 val images in data/div2k_prepared


In [5]:
!python -m pytest -q

........................................                                 [100%]
40 passed in 9.14s


## 2. Uniform-sampling baseline (Experiments A and B)
All four attacks (noise, blur, brightness, resize) are sampled with probability 0.25. Validation runs after every epoch, under every attack separately. Expected time on a T4 is roughly 20-30 minutes.

In [6]:
!python scripts/train.py --config configs/baseline.yaml

[epoch 1/60] loss=0.7065 psnr=28.47 ssim=0.892 bit_acc: clean=0.507 noise=0.506 blur=0.506 brightness=0.508 resize=0.507 (35s)
[epoch 2/60] loss=0.5895 psnr=18.17 ssim=0.681 bit_acc: clean=0.850 noise=0.847 blur=0.848 brightness=0.853 resize=0.848 (32s)
Train bit accuracy 0.960: image-loss ramp starts at epoch 4
[epoch 3/60] loss=0.1759 psnr=19.41 ssim=0.756 bit_acc: clean=0.998 noise=0.998 blur=0.997 brightness=0.995 resize=0.998 (33s)
[epoch 4/60] loss=0.0472 psnr=24.70 ssim=0.876 bit_acc: clean=0.999 noise=0.998 blur=0.998 brightness=0.991 resize=0.999 (35s)
[epoch 5/60] loss=0.0397 psnr=24.99 ssim=0.899 bit_acc: clean=1.000 noise=1.000 blur=1.000 brightness=0.997 resize=1.000 (36s)
[epoch 6/60] loss=0.0409 psnr=25.49 ssim=0.914 bit_acc: clean=0.999 noise=0.999 blur=0.999 brightness=0.994 resize=0.999 (35s)
[epoch 7/60] loss=0.0390 psnr=27.22 ssim=0.930 bit_acc: clean=1.000 noise=0.999 blur=0.999 brightness=0.994 resize=0.999 (35s)
[epoch 8/60] loss=0.0408 psnr=27.79 ssim=0.927 bit_

In [7]:
# Standalone evaluation from the saved checkpoint: clean + every attack, PSNR/SSIM (Table 1 data)
!python scripts/evaluate.py --checkpoint runs/baseline_uniform/best.pt --out results/baseline_eval.json

Evaluated 400 validation crops from runs/baseline_uniform/best.pt (epoch 50)
PSNR 32.97 dB | SSIM 0.9294
  clean        bit_acc 1.0000  msg_acc 1.0000
  noise        bit_acc 0.9990  msg_acc 0.9675
  blur         bit_acc 0.9996  msg_acc 0.9875
  brightness   bit_acc 0.9979  msg_acc 0.9850
  resize       bit_acc 0.9988  msg_acc 0.9700
Worst case: brightness (0.9979)
Wrote results/baseline_eval.json and results/baseline_eval.csv


## 3. FGAS probability update from the baseline (Experiment C)
failure = 1 - bit accuracy. p = min_prob + (1 - K * min_prob) * failure / sum(failure), with min_prob = 0.1.

In [8]:
!python scripts/fgas_update.py --eval-json results/baseline_eval.json --config configs/fgas.yaml --out results/fgas_update.json

attack       val_bit_acc   failure  old_prob  new_prob
noise             0.9990    0.0010    0.2500    0.2300
blur              0.9996    0.0004    0.2500    0.1500
brightness        0.9979    0.0021    0.2500    0.3700
resize            0.9988    0.0012    0.2500    0.2500
Wrote results/fgas_update.json and results/fgas_update.csv


## 4. FGAS integration check
A short run that starts from the baseline weights. It applies an FGAS update first, then trains with the new distribution. Compare `prob_*` and `count_*` in `metrics.csv` with the uniform baseline.

In [9]:
!python scripts/train.py --config configs/fgas.yaml --set train.epochs=2 experiment.name=fgas_integration_check

Initialized model weights from runs/baseline_uniform/best.pt
FGAS update (after epoch 0):
attack       val_bit_acc   failure  old_prob  new_prob
noise             0.9990    0.0010    0.2500    0.2300
blur              0.9996    0.0004    0.2500    0.1500
brightness        0.9979    0.0021    0.2500    0.3700
resize            0.9988    0.0012    0.2500    0.2500
[epoch 1/2] loss=0.0173 psnr=33.05 ssim=0.926 bit_acc: clean=1.000 noise=0.997 blur=0.998 brightness=0.997 resize=0.994 (35s)
[epoch 2/2] loss=0.0180 psnr=34.18 ssim=0.942 bit_acc: clean=0.999 noise=0.997 blur=0.998 brightness=0.997 resize=0.996 (36s)


## 5. Collect results and copy to Drive

In [10]:
import shutil, json, pathlib
out = pathlib.Path('results')
for run, files in {
    'baseline_uniform': ['metrics.csv', 'config.yaml', 'run_info.json', 'best_eval.json', 'best.pt'],
    'fgas_integration_check': ['metrics.csv', 'config.yaml', 'run_info.json', 'fgas_history.json'],
}.items():
    (out / run).mkdir(parents=True, exist_ok=True)
    for name in files:
        shutil.copy(pathlib.Path('runs') / run / name, out / run / name)

stamp = json.load(open('results/baseline_uniform/run_info.json'))['started_at'].replace(' ', '_').replace(':', '')
dest = f'{DRIVE_DIR}/colab_runs/{stamp}'
shutil.copytree('results', f'{dest}/results', dirs_exist_ok=True)
shutil.copytree('runs', f'{dest}/runs', dirs_exist_ok=True)
print('Copied results/ and runs/ to', dest)
!ls -R results

Copied results/ and runs/ to /content/drive/MyDrive/CMPE258_Team7/colab_runs/2026-10-20_185720
results:
baseline_eval.csv   baseline_uniform	    fgas_update.csv
baseline_eval.json  fgas_integration_check  fgas_update.json

results/baseline_uniform:
best_eval.json	best.pt  config.yaml  metrics.csv  run_info.json

results/fgas_integration_check:
config.yaml  fgas_history.json	metrics.csv  run_info.json


In [11]:
ev = json.load(open('results/baseline_eval.json'))
print(f"Baseline (epoch {ev['epoch']}): PSNR {ev['image_quality']['psnr']:.2f} dB, SSIM {ev['image_quality']['ssim']:.4f}")
for a, v in ev['attacks'].items():
    print(f"  {a:<11} bit_acc {v['bit_acc']:.4f}  msg_acc {v['msg_acc']:.4f}")
print(open('results/fgas_update.csv').read())

Baseline (epoch 50): PSNR 32.97 dB, SSIM 0.9294
  clean       bit_acc 1.0000  msg_acc 1.0000
  noise       bit_acc 0.9990  msg_acc 0.9675
  blur        bit_acc 0.9996  msg_acc 0.9875
  brightness  bit_acc 0.9979  msg_acc 0.9850
  resize      bit_acc 0.9988  msg_acc 0.9700
attack,val_bit_acc,failure,old_prob,new_prob
noise,0.9990,0.0010,0.2500,0.2300
blur,0.9996,0.0004,0.2500,0.1500
brightness,0.9979,0.0021,0.2500,0.3700
resize,0.9988,0.0012,0.2500,0.2500



## Optional: Experiment D, FGAS vs uniform continuation (Taehun)
Both runs start from the same baseline weights and train for the same number of epochs. Only the attack sampling differs. Uncomment both lines to run them (about 10 minutes each on a T4).

In [12]:
# !python scripts/train.py --config configs/baseline_continue.yaml
# !python scripts/train.py --config configs/fgas.yaml
# !python scripts/evaluate.py --checkpoint runs/baseline_continue/best.pt --out results/baseline_continue_eval.json
# !python scripts/evaluate.py --checkpoint runs/fgas_continue/best.pt --out results/fgas_continue_eval.json